In [ ]:
!pip install pandas numpy openpyxl xlsxwriter

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Annual Energy Statement.xlsx to Annual Energy Statement (1).xlsx


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load the file
file_path = "Annual Energy Statement.xlsx"
xls = pd.ExcelFile('/content/Annual Energy Statement (1).xlsx')

In [ ]:
#  To See the sheet names
print(xls.sheet_names)

['Summary', 'Level1', 'Level2', 'Level3']


In [ ]:
# Skip first 4 metadata rows
df1 = xls.parse("Level1", skiprows=4)

In [ ]:
# Renaming columns for clarity
df1.columns = ["Metric", "Units", "Value"]

In [ ]:
# Droping rows where Metric is missing
df1 = df1.dropna(subset=["Metric"])

In [ ]:
# Converting Value column to numeric
df1["Value"] = pd.to_numeric(df1["Value"], errors="coerce")

In [ ]:
df2 = xls.parse("Level2")

In [ ]:
# To Drop empty extra columns
df2 = df2.loc[:, ~df2.columns.str.contains("Unnamed")]
# Drop rows with less than 3 valid entries
df2 = df2.dropna(thresh=3)


In [ ]:
# Converting wide format (years into columns) to long format for understanding
df2_melted = pd.melt(df2, id_vars=["Fossil CO2 target", "Units"],
                     var_name="Year", value_name="Value")

In [ ]:
df2_melted.columns = ["Metric", "Units", "Year", "Value"]
df2_melted["Value"] = pd.to_numeric(df2_melted["Value"], errors="coerce")

In [ ]:
df3 = xls.parse("Level3")
# To remove completely empty rows
df3 = df3.dropna(how="all")
df3.columns = ["Metric", "Units", "2021", "2022"]

In [ ]:
# Reshape from wide to long format
df3_melted = pd.melt(df3, id_vars=["Metric", "Units"],var_name="Year", value_name="Value")

In [ ]:
df3_melted["Value"] = pd.to_numeric(df3_melted["Value"], errors="coerce")


In [ ]:
#Save Cleaned Data into a New Excel File
with pd.ExcelWriter("Cleaned_Energy_Data.xlsx", engine="xlsxwriter") as writer:
    df1.to_excel(writer, sheet_name="Clean_Level1", index=False)
    df2_melted.to_excel(writer, sheet_name="Clean_Level2", index=False)
    df3_melted.to_excel(writer, sheet_name="Clean_Level3", index=False)

In [ ]:
files.download("Cleaned_Energy_Data.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>